# 16 - Fine-tuned FinBERT Dış Test Değerlendirmesi

Bu notebook **08_finetune_finbert_target_dataset.ipynb** ile oluşturulan:

`checkpoints/financial_sentiment_multi_model/finbert_target_finetuned_seed42/final_model`

modelini yeniden eğitmeden üç dış test kümesinde değerlendirir:

1. **S&P 500**
2. **Synthetic financial news**
3. **Reuters 5000**

Üretilen temel çıktılar:

- Accuracy
- Macro Precision
- Macro Recall
- Macro-F1
- Weighted-F1
- Sınıf bazlı F1
- Confusion matrix
- Normalize confusion matrix
- Örnek bazlı prediction CSV
- Tüm dış testleri tek tabloda birleştiren summary CSV

> Bu notebook training yapmaz; yalnızca kayıtlı fine-tuned FinBERT checkpoint'i üzerinde inference yapar.


<!-- thesis-review-note -->
## Çalışma Notu

- Amaç: Fine-tuned FinBERT modelini S&P 500, sentetik ve Reuters dis testlerinde degerlendirir.
- Girdi: 08 numarali notebook final modeli ve uc dis test seti.
- Çıktı ve değerlendirme: Fine-tuned FinBERT sonuclarini diger modellerle dis veri bazinda karsilastirilabilir hale getirir.
- Sıra notu: Notebook numarasi deney akışındaki yerini gösterir; önceki numaralar tamamlanmadan kalıcı sonuç yorumları güncellenmemelidir.
- Sonuç güvenliği: Bu dosyadaki mevcut output hücreleri ve kalıcı sonuç dosyaları korunur.


In [1]:
# ============================================================
# 1) IMPORTLAR VE AYARLAR
# ============================================================

from thesis_utils import PROJECT_ROOT, PREVIEW_ROWS, paths

from pathlib import Path
import gc
import json
import re
import warnings

import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    f1_score,
    classification_report,
    confusion_matrix,
)

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)
pd.set_option("display.max_colwidth", 180)

PROJECT_DIR = PROJECT_ROOT

CHECKPOINT_ROOT = paths.MODEL_CHECKPOINT_ROOT
RUN_DIR = CHECKPOINT_ROOT / "finbert_target_finetuned_seed42"
MODEL_DIR = RUN_DIR / "final_model"
RESULTS_DIR = paths.MODEL_RESULTS_ROOT / "finbert_target_finetuned_seed42" / "external_evaluation"

SP500_DIR = paths.SP500_HUMAN_REVIEW_BATCHES_DIR
SYNTHETIC_DIR = paths.SYNTHETIC_NEWS_DIR
REUTERS_DIR = paths.REUTERS_ANNOTATION_DIR

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

LABELS = ["negative", "neutral", "positive"]
LABEL2ID = {"negative": 0, "neutral": 1, "positive": 2}
ID2LABEL = {0: "negative", 1: "neutral", 2: "positive"}

MAX_LENGTH = 128
BATCH_SIZE = 32

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PROJECT_DIR :", PROJECT_DIR)
print("MODEL_DIR   :", MODEL_DIR)
print("RESULTS_DIR :", RESULTS_DIR)
print("DEVICE      :", DEVICE)

for name, p in {
    "MODEL_DIR": MODEL_DIR,
    "SP500_DIR": SP500_DIR,
    "SYNTHETIC_DIR": SYNTHETIC_DIR,
    "REUTERS_DIR": REUTERS_DIR,
}.items():
    print(f"{name:14s} exists:", Path(p).exists(), "|", p)

if not MODEL_DIR.exists():
    raise FileNotFoundError(
        "Fine-tuned FinBERT final_model bulunamadı. "
        "Önce 08_finetune_finbert_target_dataset.ipynb tamamlanmış olmalı.\n"
        f"Beklenen yol: {MODEL_DIR}"
    )


PROJECT_DIR : D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis
MODEL_DIR   : D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\checkpoints\financial_sentiment_multi_model\finbert_target_finetuned_seed42\final_model
RESULTS_DIR : D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\checkpoints\financial_sentiment_multi_model\finbert_target_finetuned_seed42\external_evaluation
DEVICE      : cpu
MODEL_DIR      exists: True | D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\checkpoints\financial_sentiment_multi_model\finbert_target_finetuned_seed42\final_model
SP500_DIR      exists: True | D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db\annotations\sp500_human_review_batches
SYNTHETIC_DIR  exists: True | D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db\evaluation\synthetic_financial_news
REUTERS_DIR    exists: True | D:\serkan.kaymak\financial_sentiment_thes

In [2]:
from thesis_utils.data_io import (
    clean_string_series,
    make_unique_columns,
    normalize_column_name,
)

normalize_colname = normalize_column_name

# ============================================================
# 2) ORTAK YARDIMCI FONKSİYONLAR
# ============================================================

def choose_text_column(df):
    for col in ["text_en", "text", "source_text", "headline", "title"]:
        if col in df.columns:
            s = clean_string_series(df[col])
            if s.notna().any():
                return col
    raise ValueError(f"Metin kolonu bulunamadı. Kolonlar: {df.columns.tolist()}")


def prepare_eval_df(df, gold_candidates, dataset_name):
    df = df.copy()

    text_col = choose_text_column(df)
    df["eval_text"] = clean_string_series(df[text_col])

    df["gold_label"] = pd.NA

    for col in gold_candidates:
        if col in df.columns:
            candidate = clean_string_series(df[col]).str.lower()
            df["gold_label"] = df["gold_label"].fillna(candidate)

    df = df[
        df["eval_text"].notna()
        & df["gold_label"].isin(LABELS)
    ].copy().reset_index(drop=True)

    if len(df) == 0:
        raise ValueError(
            f"{dataset_name}: değerlendirilebilir satır bulunamadı. "
            f"Gold adayları: {gold_candidates}"
        )

    df["gold_id"] = df["gold_label"].map(LABEL2ID).astype(int)

    if "sample_id" not in df.columns:
        df.insert(0, "sample_id", [f"{dataset_name}_{i:06d}" for i in range(len(df))])
    else:
        df["sample_id"] = clean_string_series(df["sample_id"])
        missing = df["sample_id"].isna()
        df.loc[missing, "sample_id"] = [
            f"{dataset_name}_{i:06d}" for i in df.index[missing]
        ]

    print("\n" + "=" * 100)
    print(dataset_name.upper())
    print("=" * 100)
    print("N:", len(df))
    print(df["gold_label"].value_counts().reindex(LABELS))

    return df


In [3]:
from thesis_utils.data_io import extract_batch_number, find_header_row as _find_header_row

extract_batch_no = extract_batch_number

def find_header_row(raw_df, required_cols=("annotation_id", "sample_id", "text_en")):
    header_row = _find_header_row(
        raw_df,
        required_cols=required_cols,
        min_hits=2,
        max_scan_rows=10,
    )
    return 0 if header_row is None else header_row

# ============================================================
# 3) S&P 500 VERİSİNİ YÜKLE
# 05_evaluate_sp500_finetuned_models.ipynb ile AYNI okuma mantığı.
#
# ÖNEMLİ:
# İlk sürüm pd.read_excel(..., header=0) kullandığı için başlık satırı
# ilk satırda olmayan bazı batch dosyalarını atlayabiliyordu.
# Bu sürüm 05 notebook'undaki gibi header satırını otomatik bulur.
# Beklenen toplam: 106 batch x 10 örnek = 1060 satır.
# ============================================================

SP500_BATCH_PATTERN = "SP500_annotation_batch_*.xlsx"


def read_sp500_annotation_excel(path):
    path = Path(path)

    raw = pd.read_excel(path, header=None, dtype=object)
    header_row = find_header_row(raw)

    df = pd.read_excel(path, header=header_row, dtype=object)
    df.columns = [normalize_colname(c) for c in df.columns]
    df = df.dropna(how="all").copy()

    df["source_file"] = path.name
    df["batch_no"] = extract_batch_no(path)
    df["detected_header_row"] = header_row

    # Özet / Unnamed kolonları değerlendirmeye dahil etme
    drop_cols = []
    for c in df.columns:
        lc = str(c).lower()
        if lc.startswith("unnamed"):
            drop_cols.append(c)
        if c in ["Özet", "Değer", "Ozet", "Deger"]:
            drop_cols.append(c)

    df = df.drop(columns=list(set(drop_cols)), errors="ignore")

    return df


def load_sp500():
    if not SP500_DIR.exists():
        raise FileNotFoundError(f"S&P 500 annotation klasörü bulunamadı: {SP500_DIR}")

    files = sorted(
        [
            p for p in SP500_DIR.glob(SP500_BATCH_PATTERN)
            if p.is_file() and not p.name.startswith("~$")
        ],
        key=lambda p: (extract_batch_no(p), p.name.lower())
    )

    # Fallback: ana klasörde doğrudan bulunamazsa recursive ara
    if not files:
        files = sorted(
            [
                p for p in SP500_DIR.rglob(SP500_BATCH_PATTERN)
                if p.is_file() and not p.name.startswith("~$")
            ],
            key=lambda p: (extract_batch_no(p), p.name.lower())
        )

    if not files:
        raise FileNotFoundError(
            f"S&P 500 annotation batch dosyası bulunamadı: {SP500_DIR}\n"
            f"Pattern: {SP500_BATCH_PATTERN}"
        )

    print("Bulunan S&P 500 batch sayısı:", len(files))
    print("Beklenen (mevcut proje çıktısına göre): 106")

    frames = []
    failed_files = []
    header_shift_files = []

    for p in tqdm(files, desc="S&P 500 batchleri okunuyor"):
        try:
            temp = read_sp500_annotation_excel(p)

            if "annotation_id" not in temp.columns:
                failed_files.append((p.name, "annotation_id kolonu bulunamadı"))
                continue

            header_row = int(temp["detected_header_row"].iloc[0]) if len(temp) else 0
            if header_row != 0:
                header_shift_files.append((p.name, header_row))

            frames.append(temp)

        except Exception as e:
            failed_files.append((p.name, repr(e)))

    if not frames:
        raise ValueError("S&P 500 için uygun annotation dosyası okunamadı.")

    df = pd.concat(frames, ignore_index=True)

    if "annotation_id" in df.columns:
        df["annotation_id"] = clean_string_series(df["annotation_id"])
        df = df[df["annotation_id"].notna()].copy()

    print("Okunan batch sayısı:", len(frames))
    print("Ham annotation satırı:", len(df))
    print("Başlığı 1. satırda olmayan batch sayısı:", len(header_shift_files))

    if header_shift_files:
        print("Başlık kayması olan batchler:")
        for item in header_shift_files:
            print(" -", item)

    if failed_files:
        print("UYARI - okunamayan / atlanan batchler:")
        for item in failed_files:
            print(" -", item)

    eval_df = prepare_eval_df(
        df,
        gold_candidates=["final_label", "chatgpt_label"],
        dataset_name="sp500_external"
    )

    # Projedeki 05 notebook ve önceki tez tablolarıyla tutarlılık kontrolü
    if len(files) == 106 and len(eval_df) != 1060:
        raise ValueError(
            f"S&P 500 satır sayısı {len(eval_df)} çıktı; beklenen 1060. "
            "Teze sonuç aktarmadan önce batch/etiket farkı incelenmeli."
        )

    return eval_df


sp500_df = load_sp500()
display(sp500_df.head(PREVIEW_ROWS))


Bulunan S&P 500 batch sayısı: 106
Beklenen (mevcut proje çıktısına göre): 106


S&P 500 batchleri okunuyor:   0%|          | 0/106 [00:00<?, ?it/s]

Okunan batch sayısı: 106
Ham annotation satırı: 1060
Başlığı 1. satırda olmayan batch sayısı: 3
Başlık kayması olan batchler:
 - ('SP500_annotation_batch_102.xlsx', 3)
 - ('SP500_annotation_batch_107.xlsx', 2)
 - ('SP500_annotation_batch_109.xlsx', 2)

SP500_EXTERNAL
N: 1060
gold_label
negative    323
neutral     339
positive    398
Name: count, dtype: int64


,annotation_id,sample_id,date,text_en,text_tr,finbert_label,finbert_confidence,chatgpt_label,chatgpt_confidence,chatgpt_reason_tr,finbert_correctness,finbert_correctness_note,source_file,batch_no,detected_header_row,final_label,eval_text,gold_label,gold_id
0,SP500_ANN_0001,SP500_HEAD_000004,2008-01-03,"U.S. Stocks Higher After Economic Data, Monsanto Outlook",ABD hisseleri ekonomik veriler ve Monsanto görünümü sonrası yükseldi.,positive,0.861627,positive,high,ABD hisseleri yükseliyor; piyasa açısından olumlu sinyal.,correct,FinBERT etiketi final etiket ile aynı.,SP500_annotation_batch_001.xlsx,1,0,NaN,"U.S. Stocks Higher After Economic Data, Monsanto Outlook",positive,2
1,SP500_ANN_0002,SP500_HEAD_016918,2023-12-15,Stock Market Outlook 2024: Rare Bullish Signal Says S&P 500 Will Soar 20%,2024 borsa görünümü: Nadir bir boğa sinyali S&P 500'ün %20 yükseleceğini söylüyor.,positive,0.881073,positive,high,Bullish sinyal ve S&P 500'de güçlü yükseliş beklentisi var.,correct,FinBERT etiketi final etiket ile aynı.,SP500_annotation_batch_001.xlsx,1,0,NaN,Stock Market Outlook 2024: Rare Bullish Signal Says S&P 500 Will Soar 20%,positive,2
2,SP500_ANN_0003,SP500_HEAD_013289,2023-03-20,Federal Reserve Rate Hike Odds Grow As Bank-Crisis Fears Ebb; S&P 500 Rises,Banka krizi korkuları azalırken Fed faiz artırımı olasılığı yükseliyor; S&P 500 yükseliyor.,negative,0.625434,positive,medium,Başlık karışık olsa da banka krizi korkularının azalması ve S&P 500'ün yükselmesi piyasa açısından olumlu baskın sinyal veriyor.,wrong,FinBERT negative demiş; final etiket positive. Muhtemelen 'rate hike odds grow' kısmına fazla ağırlık verdi.,SP500_annotation_batch_001.xlsx,1,0,NaN,Federal Reserve Rate Hike Odds Grow As Bank-Crisis Fears Ebb; S&P 500 Rises,positive,2


In [4]:
from thesis_utils.data_io import natural_sort_key

# ============================================================
# 4) SENTETİK VERİYİ YÜKLE
# 06_evaluate_synthetic_finetuned_models.ipynb ile aynı klasörü kullanır.
# ============================================================

def load_synthetic():
    if not SYNTHETIC_DIR.exists():
        raise FileNotFoundError(f"Sentetik veri klasörü bulunamadı: {SYNTHETIC_DIR}")

    patterns = [
        "positive_batch_*.xlsx",
        "negative_batch_*.xlsx",
        "neutral_batch_*.xlsx",
    ]

    files = []

    for pattern in patterns:
        files.extend(
            [p for p in SYNTHETIC_DIR.glob(pattern) if not p.name.startswith("~$")]
        )

    files = sorted(set(files), key=natural_sort_key)

    if not files:
        raise FileNotFoundError(f"Sentetik Excel dosyası bulunamadı: {SYNTHETIC_DIR}")

    frames = []

    for p in files:
        xls = pd.ExcelFile(p)
        sheet_name = "Data" if "Data" in xls.sheet_names else xls.sheet_names[0]

        temp = pd.read_excel(p, sheet_name=sheet_name)
        temp.columns = [normalize_colname(c) for c in temp.columns]
        temp = temp.dropna(how="all").copy()

        if "label" not in temp.columns:
            print("UYARI - label kolonu yok, atlandı:", p.name)
            continue

        temp["source_file"] = p.name
        frames.append(temp)

    if not frames:
        raise ValueError("Sentetik veri için uygun dosya okunamadı.")

    df = pd.concat(frames, ignore_index=True)

    return prepare_eval_df(
        df,
        gold_candidates=["label"],
        dataset_name="synthetic_external"
    )


synthetic_df = load_synthetic()
display(synthetic_df.head(PREVIEW_ROWS))



SYNTHETIC_EXTERNAL
N: 3000
gold_label
negative    1000
neutral     1000
positive    1000
Name: count, dtype: int64


,annotation_id,sample_id,date,text_en,text_tr,label,label_id,label_confidence,reason_tr,sector,topic,split,source_type,created_at,source_file,synthetic_note,eval_text,gold_label,gold_id
0,SYN_NEG_ANN_0821,SYN_NEG_HEAD_000821,2026-04-01 00:00:00,A cybersecurity vendor reported slower billings growth as large enterprise deals took longer to close.,"Bir siber güvenlik sağlayıcısı, büyük kurumsal anlaşmaların kapanmasının uzamasıyla faturalama büyümesinin yavaşladığını bildirdi.",negative,0,high,Faturalama büyümesindeki yavaşlama gelir momentumunu olumsuz etkiler.,Information Technology,billings_slowdown,train,synthetic,2026-05-25 17:00:00,negative_batch_000.xlsx,NaN,A cybersecurity vendor reported slower billings growth as large enterprise deals took longer to close.,negative,0
1,SYN_NEG_ANN_0822,SYN_NEG_HEAD_000822,2026-04-02 00:00:00,A consumer finance company increased reserves after delinquency rates rose in its credit card portfolio.,"Bir tüketici finansmanı şirketi, kredi kartı portföyünde gecikme oranlarının yükselmesi sonrası rezervlerini artırdı.",negative,0,high,Gecikme oranı ve rezerv artışı varlık kalitesi açısından negatiftir.,Financials,delinquency_increase,train,synthetic,2026-05-25 17:00:00,negative_batch_000.xlsx,NaN,A consumer finance company increased reserves after delinquency rates rose in its credit card portfolio.,negative,0
2,SYN_NEG_ANN_0823,SYN_NEG_HEAD_000823,2026-04-03 00:00:00,A hospital operator cut earnings guidance after labor expenses remained above management's expectations.,"Bir hastane işletmecisi, işçilik giderlerinin yönetim beklentilerinin üzerinde kalması sonrası kâr beklentisini düşürdü.",negative,0,high,Yüksek işçilik giderleri ve beklenti indirimi kârlılık için negatiftir.,Health Care,guidance_cut,train,synthetic,2026-05-25 17:00:00,negative_batch_000.xlsx,NaN,A hospital operator cut earnings guidance after labor expenses remained above management's expectations.,negative,0


In [5]:
from thesis_utils.data_io import (
    find_header_row,
    read_raw_table_no_header,
)

read_raw_no_header = read_raw_table_no_header

# ============================================================
# 5) REUTERS 5000 VERİSİNİ YÜKLE
# 07_evaluate_reuters_finetuned_models.ipynb mantığıyla robust okur.
# ============================================================

def read_reuters_file(path):
    raw = read_raw_no_header(path).dropna(how="all").copy()
    header_idx = find_header_row(raw)

    if header_idx is None:
        if path.suffix.lower() == ".xlsx":
            df = pd.read_excel(path, dtype=object)
        else:
            df = pd.read_csv(path, encoding="utf-8-sig", dtype=object)
        df.columns = make_unique_columns(df.columns)
    else:
        header = raw.iloc[header_idx].tolist()
        df = raw.iloc[header_idx + 1:].copy()
        df.columns = make_unique_columns(header)

    df = df.dropna(how="all").dropna(axis=1, how="all").copy()

    if "annotation_id" not in df.columns:
        return None

    if "text_en" not in df.columns:
        df["text_en"] = pd.NA

    df["text_en"] = clean_string_series(df["text_en"])

    for fallback in ["text", "source_text", "headline", "title"]:
        if fallback in df.columns:
            df["text_en"] = df["text_en"].fillna(clean_string_series(df[fallback]))

    if "final_label" not in df.columns:
        df["final_label"] = pd.NA
    if "chatgpt_label" not in df.columns:
        df["chatgpt_label"] = pd.NA

    df["source_file"] = path.name
    df["batch_no"] = extract_batch_no(path)

    return df


def load_reuters():
    if not REUTERS_DIR.exists():
        raise FileNotFoundError(f"Reuters annotation klasörü bulunamadı: {REUTERS_DIR}")

    files = []
    for pat in [
        "*annotation_batch_*.xlsx",
        "*annotation_batch_*.csv",
        "*ANNOTATION_BATCH_*.xlsx",
        "*ANNOTATION_BATCH_*.csv",
    ]:
        files.extend(REUTERS_DIR.rglob(pat))

    files = sorted(
        {p for p in files if p.is_file() and not p.name.startswith("~$")},
        key=lambda p: (extract_batch_no(p), p.name.lower())
    )

    if not files:
        raise FileNotFoundError(f"Reuters batch dosyası bulunamadı: {REUTERS_DIR}")

    frames = []

    for p in files:
        try:
            temp = read_reuters_file(p)
            if temp is not None and len(temp):
                frames.append(temp)
        except Exception as e:
            print("UYARI - okunamadı:", p, "|", e)

    if not frames:
        raise ValueError("Reuters için uygun annotation dosyası okunamadı.")

    df = pd.concat(frames, ignore_index=True)

    return prepare_eval_df(
        df,
        gold_candidates=["final_label", "chatgpt_label"],
        dataset_name="reuters_external"
    )


reuters_df = load_reuters()
display(reuters_df.head(PREVIEW_ROWS))



REUTERS_EXTERNAL
N: 5000
gold_label
negative    1579
neutral     1146
positive    2275
Name: count, dtype: int64


,sample_id,annotation_id,source_dataset,source_row_index,date,date_only,text_en,finbert_score,finbert_label,finbert_confidence,primary_topic,topics,topic_confidence,financial_score,hour,n_words,text_len,final_label,chatgpt_label,source_file,batch_no,batch_id,text,text_tr,chatgpt_score,chatgpt_confidence,chatgpt_reason_tr,finbert_correctness,review_needed,row_note,finbert_match,review_note_tr,review_note,review_flag,finbert_comparison,annotation_note,label_source_type,source_text,finbert_vs_chatgpt,review_priority,needs_review,finbert_alignment,source_batch,row_no,eval_text,gold_label,gold_id
0,reuters_external_000000,REUTERS_ANN_00001,NaN,NaN,2007-04-12,NaN,s.africa watchdog to probe gold fields bid report,-1,negative,0.683211,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,negative,REUTERS_annotation_batch_001.xlsx,1,REUTERS_annotation_batch_001,s.africa watchdog to probe gold fields bid report,Güney Afrika rekabet kurumu Gold Fields teklifi raporunu inceleyecek,-1,0.86,Bir satın alma teklifinin rekabet kurumu tarafından incelenmesi düzenleyici belirsizlik ve işlem riski yaratır.,same,no,FinBERT etiketi bağlamla uyumlu; 'watchdog to probe' ifadesi düzenleyici soruşturma/inceleme riskini negatif yakalamış.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,s.africa watchdog to probe gold fields bid report,negative,0
1,reuters_external_000001,REUTERS_ANN_00002,NaN,NaN,2007-04-26,NaN,new barbie girls sashay into view with mp-3,0,neutral,0.847032,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,positive,REUTERS_annotation_batch_001.xlsx,1,REUTERS_annotation_batch_001,new barbie girls sashay into view with mp-3,Yeni Barbie Girls MP3 ile sahneye çıkıyor,1,0.76,Yeni MP3 özellikli ürün lansmanı ürün yeniliği ve satış potansiyeli açısından sınırlı olumlu sinyal verir.,different,yes,FinBERT bunu nötr görmüş olabilir çünkü başlık finansal metrik içermiyor; ancak yeni ürün tanıtımı pozitif ürün/growth sinyalidir.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,new barbie girls sashay into view with mp-3,positive,2
2,reuters_external_000002,REUTERS_ANN_00003,NaN,NaN,2006-12-17,NaN,"stocks await data, mergers and santa",0,neutral,0.865116,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,neutral,REUTERS_annotation_batch_001.xlsx,1,REUTERS_annotation_batch_001,"stocks await data, mergers and santa","Hisseler veri, birleşmeler ve Noel Baba rallisini bekliyor",0,0.82,Piyasa bekleyişi anlatılıyor; veri veya birleşmelerin net sonucu başlıkta yok.,same,no,FinBERT etiketi bağlamla uyumlu; 'await' bekle-gör ve yönsüz piyasa tonudur.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"stocks await data, mergers and santa",neutral,1


In [6]:
# ============================================================
# 6) MODELİ BİR KEZ YÜKLE
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)

model.config.id2label = ID2LABEL.copy()
model.config.label2id = LABEL2ID.copy()

model.to(DEVICE)
model.eval()

print("Model loaded:", MODEL_DIR)
print("id2label:", model.config.id2label)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model loaded: D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\checkpoints\financial_sentiment_multi_model\finbert_target_finetuned_seed42\final_model
id2label: {0: 'negative', 1: 'neutral', 2: 'positive'}


In [7]:
# ============================================================
# 7) INFERENCE + METRİK + KAYIT FONKSİYONLARI
# ============================================================

@torch.no_grad()
def predict_texts(texts, batch_size=BATCH_SIZE, max_length=MAX_LENGTH):
    pred_ids_all = []
    confidences = []
    probs_all = []

    texts = [str(x) for x in texts]

    for start in tqdm(range(0, len(texts), batch_size), desc="Fine-tuned FinBERT inference"):
        batch = texts[start:start + batch_size]

        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        enc = {k: v.to(DEVICE) for k, v in enc.items()}

        outputs = model(**enc)
        probs = torch.softmax(outputs.logits, dim=-1).detach().cpu().numpy()

        pred_ids = probs.argmax(axis=1)

        pred_ids_all.extend(pred_ids.astype(int).tolist())
        confidences.extend(probs.max(axis=1).astype(float).tolist())
        probs_all.append(probs)

    probs_all = np.vstack(probs_all)

    return pd.DataFrame({
        "prediction_id": pred_ids_all,
        "prediction": [ID2LABEL[int(x)] for x in pred_ids_all],
        "confidence": confidences,
        "prob_negative": probs_all[:, 0],
        "prob_neutral": probs_all[:, 1],
        "prob_positive": probs_all[:, 2],
    })


def evaluate_dataset(df, dataset_name):
    pred_df = predict_texts(df["eval_text"].tolist())

    y_true = df["gold_id"].astype(int).to_numpy()
    y_pred = pred_df["prediction_id"].astype(int).to_numpy()

    acc = accuracy_score(y_true, y_pred)

    p_macro, r_macro, f_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[0, 1, 2], average="macro", zero_division=0
    )

    _, _, f_weighted, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[0, 1, 2], average="weighted", zero_division=0
    )

    p_cls, r_cls, f_cls, support_cls = precision_recall_fscore_support(
        y_true, y_pred, labels=[0, 1, 2], average=None, zero_division=0
    )

    metrics = {
        "dataset": dataset_name,
        "model": "fine_tuned_finbert_seed42",
        "n": int(len(df)),
        "accuracy": float(acc),
        "precision_macro": float(p_macro),
        "recall_macro": float(r_macro),
        "f1_macro": float(f_macro),
        "f1_weighted": float(f_weighted),
        "f1_negative": float(f_cls[0]),
        "f1_neutral": float(f_cls[1]),
        "f1_positive": float(f_cls[2]),
        "support_negative": int(support_cls[0]),
        "support_neutral": int(support_cls[1]),
        "support_positive": int(support_cls[2]),
    }

    report_dict = classification_report(
        y_true,
        y_pred,
        labels=[0, 1, 2],
        target_names=LABELS,
        output_dict=True,
        zero_division=0
    )

    report_text = classification_report(
        y_true,
        y_pred,
        labels=[0, 1, 2],
        target_names=LABELS,
        digits=4,
        zero_division=0
    )

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])

    row_totals = cm.sum(axis=1, keepdims=True)
    cm_norm = np.divide(
        cm,
        row_totals,
        out=np.zeros_like(cm, dtype=float),
        where=row_totals != 0
    )

    cm_df = pd.DataFrame(
        cm,
        index=[f"true_{x}" for x in LABELS],
        columns=[f"pred_{x}" for x in LABELS]
    )

    cm_norm_df = pd.DataFrame(
        cm_norm,
        index=[f"true_{x}" for x in LABELS],
        columns=[f"pred_{x}" for x in LABELS]
    )

    keep_cols = [
        "sample_id",
        "annotation_id",
        "date",
        "source_file",
        "eval_text",
        "gold_label",
        "gold_id",
    ]
    keep_cols = [c for c in keep_cols if c in df.columns]

    result_df = pd.concat(
        [df[keep_cols].reset_index(drop=True), pred_df.reset_index(drop=True)],
        axis=1
    )

    result_df["correct"] = result_df["gold_id"] == result_df["prediction_id"]

    dataset_dir = RESULTS_DIR / dataset_name
    dataset_dir.mkdir(parents=True, exist_ok=True)

    result_df.to_csv(
        dataset_dir / "predictions.csv",
        index=False,
        encoding="utf-8-sig"
    )

    cm_df.to_csv(dataset_dir / "confusion_matrix.csv", encoding="utf-8-sig")
    cm_norm_df.to_csv(
        dataset_dir / "confusion_matrix_normalized.csv",
        encoding="utf-8-sig"
    )

    pd.DataFrame(report_dict).T.to_csv(
        dataset_dir / "classification_report.csv",
        encoding="utf-8-sig"
    )

    with open(dataset_dir / "metrics.json", "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)

    print("\n" + "#" * 110)
    print("FINE-TUNED FINBERT |", dataset_name.upper())
    print("#" * 110)

    display(pd.DataFrame([metrics]).round(6))

    print("\nClassification report:")
    print(report_text)

    print("\nConfusion matrix:")
    display(cm_df)

    print("\nNormalized confusion matrix:")
    display(cm_norm_df.round(4))

    print("\nSaved:", dataset_dir)

    return metrics, result_df


In [8]:
# ============================================================
# 8) ÜÇ DIŞ TESTİ ÇALIŞTIR
# ============================================================

all_metrics = []
all_predictions = {}

for dataset_name, df in [
    ("sp500_external", sp500_df),
    ("synthetic_external", synthetic_df),
    ("reuters_external", reuters_df),
]:
    metrics, predictions = evaluate_dataset(df, dataset_name)
    all_metrics.append(metrics)
    all_predictions[dataset_name] = predictions


Fine-tuned FinBERT inference:   0%|          | 0/34 [00:00<?, ?it/s]


##############################################################################################################
FINE-TUNED FINBERT | SP500_EXTERNAL
##############################################################################################################


,dataset,model,n,accuracy,precision_macro,recall_macro,f1_macro,f1_weighted,f1_negative,f1_neutral,f1_positive,support_negative,support_neutral,support_positive
0,sp500_external,fine_tuned_finbert_seed42,1060,0.772642,0.794458,0.772053,0.77227,0.773915,0.763573,0.753769,0.799469,323,339,398



Classification report:
              precision    recall  f1-score   support

    negative     0.8790    0.6749    0.7636       323
     neutral     0.6565    0.8850    0.7538       339
    positive     0.8479    0.7563    0.7995       398

    accuracy                         0.7726      1060
   macro avg     0.7945    0.7721    0.7723      1060
weighted avg     0.7962    0.7726    0.7739      1060


Confusion matrix:


,pred_negative,pred_neutral,pred_positive
true_negative,218,77,28
true_neutral,13,300,26
true_positive,17,80,301


...


Classification report:
              precision    recall  f1-score   support

    negative     0.9000    0.5643    0.6937      1579
     neutral     0.3707    0.9154    0.5277      1146
    positive     0.8915    0.4624    0.6090      2275

    accuracy                         0.5984      5000
   macro avg     0.7207    0.6474    0.6101      5000
weighted avg     0.7748    0.5984    0.6171      5000


Confusion matrix:


,pred_negative,pred_neutral,pred_positive
true_negative,891,614,74
true_neutral,43,1049,54
true_positive,56,1167,1052



Normalized confusion matrix:


,pred_negative,pred_neutral,pred_positive
true_negative,0.5643,0.3889,0.0469
true_neutral,0.0375,0.9154,0.0471
true_positive,0.0246,0.5130,0.4624



Saved: D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\checkpoints\financial_sentiment_multi_model\finbert_target_finetuned_seed42\external_evaluation\reuters_external


In [9]:
# ============================================================
# 9) TOPLU ÖZET
# ============================================================

summary_df = pd.DataFrame(all_metrics)

summary_cols = [
    "dataset",
    "model",
    "n",
    "accuracy",
    "precision_macro",
    "recall_macro",
    "f1_macro",
    "f1_weighted",
    "f1_negative",
    "f1_neutral",
    "f1_positive",
]

summary_df = summary_df[summary_cols].copy()

summary_path = RESULTS_DIR / "finetuned_finbert_external_summary.csv"
summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

print("\n" + "=" * 120)
print("FINE-TUNED FINBERT - EXTERNAL TEST SUMMARY")
print("=" * 120)

display(summary_df.round(6))

print("\nTeze aktarırken özellikle kullanılacak değerler:")
for _, row in summary_df.iterrows():
    print(
        f"{row['dataset']:20s} | "
        f"Acc={row['accuracy']:.4f} | "
        f"Macro-P={row['precision_macro']:.4f} | "
        f"Macro-R={row['recall_macro']:.4f} | "
        f"Macro-F1={row['f1_macro']:.4f} | "
        f"Weighted-F1={row['f1_weighted']:.4f}"
    )

print("\nSummary saved:", summary_path)



FINE-TUNED FINBERT - EXTERNAL TEST SUMMARY


,dataset,model,n,accuracy,precision_macro,recall_macro,f1_macro,f1_weighted,f1_negative,f1_neutral,f1_positive
0,sp500_external,fine_tuned_finbert_seed42,1060,0.772642,0.794458,0.772053,0.772270,0.773915,0.763573,0.753769,0.799469
1,synthetic_external,fine_tuned_finbert_seed42,3000,0.974667,0.974985,0.974667,0.974590,0.974590,0.965764,0.987593,0.970414
2,reuters_external,fine_tuned_finbert_seed42,5000,0.598400,0.720732,0.647352,0.610098,0.617080,0.693655,0.527666,0.608973



Teze aktarırken özellikle kullanılacak değerler:
sp500_external       | Acc=0.7726 | Macro-P=0.7945 | Macro-R=0.7721 | Macro-F1=0.7723 | Weighted-F1=0.7739
synthetic_external   | Acc=0.9747 | Macro-P=0.9750 | Macro-R=0.9747 | Macro-F1=0.9746 | Weighted-F1=0.9746
reuters_external     | Acc=0.5984 | Macro-P=0.7207 | Macro-R=0.6474 | Macro-F1=0.6101 | Weighted-F1=0.6171

Summary saved: D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\checkpoints\financial_sentiment_multi_model\finbert_target_finetuned_seed42\external_evaluation\finetuned_finbert_external_summary.csv


## Notebook bittikten sonra

Bana mümkünse şu dosyayı yükle:

`outputs/finetuned_model_results/finbert_target_finetuned_seed42/external_evaluation/finetuned_finbert_external_summary.csv`

Ayrıca üç klasördeki `confusion_matrix.csv` dosyaları da tezdeki dış test tablolarını güncellemek için faydalı olur.

Bu sonuçlar geldikten sonra tezdeki şu satır:

`Fine-tuned FinBERT | 0,8512 | N/A | N/A | N/A`

S&P 500, sentetik ve Reuters sonuçlarıyla doldurulabilir.
